# Estudo de Caso: O Mecanismo de Atenção (Attention Mechanism) em NLP


---

## Objetivos de Aprendizagem
1. Compreender o problema do "gargalo de informação" em arquiteturas recorrentes (RNNs/LSTMs).
2. Dominar a matemática por trás do *Dot-Product Attention* (Cálculo de Scores, Alinhamento/Softmax e Vetor de Contexto).
3. Implementar uma camada de atenção customizada utilizando **PyTorch**.
4. Analisar e interpretar os pesos de atenção de um modelo treinado através de mapas de calor (*heatmaps*).

---
## Requisitos de Execução
Certifique-se de ter as seguintes bibliotecas instaladas em seu ambiente:
```bash
pip install torch matplotlib seaborn numpy
```

## Parte 1: O Problema do "Gargalo" nas LSTMs

Até o momento, vimos que as redes recorrentes como as LSTMs processam textos sequencialmente, palavra por palavra. Para classificar uma notícia do nosso dataset **UCI-News**, a abordagem tradicional consiste em passar o título pela LSTM e utilizar apenas o **último estado oculto ($h_{final}$)** como representação consolidada da frase.

### O Limite da Memória Comprimida
Imagine um título longo como:  
> *"Fed raises benchmark interest rate by a quarter point to counter inflation risks in the retail sector"*

No modelo tradicional, o vetor $h_{final}$ (gerado após a palavra *sector*) precisa carregar toda a informação semântica da frase. Palavras cruciais que apareceram no início, como **"Fed"**, **"raises"** e **"interest rate"**, sofrem compressão drástica e tendem a ser esquecidas ou atenuadas. Esse fenômeno é conhecido como o **Gargalo das RNNs**.

### A Solução: Mecanismo de Atenção
A atenção resolve esse problema permitindo que o classificador acesse os estados ocultos de **todos** os passos de tempo da sequência ($h_1, h_2, \dots, h_n$), e não apenas o último. O modelo aprende a ponderar dinamicamente quais palavras são mais relevantes para a tarefa de classificação corrente.

## Parte 2: A Matemática da Atenção (Dot-Product Attention)

O pipeline de cálculo da atenção global baseada em produto escalar segue três etapas fundamentais:

1. **Cálculo dos Scores de Similaridade:** 
   Calculamos o produto escalar entre o vetor de consulta (Query, que pode ser o próprio vetor de contexto do modelo ou o último estado oculto) e o estado oculto de cada palavra $h_i$ (Keys/Values):
   $$\text{score}(h_{query}, h_i) = h_{query}^T \cdot h_i$$

2. **Distribuição de Alinhamento (Softmax):** 
   Transformamos os scores brutos em probabilidades (pesos de atenção $\alpha_i$) através da função Softmax, garantindo que a soma de todos os pesos seja exatamente 1.0:
   $$\alpha_{i} = \frac{\exp(\text{score}(h_{query}, h_i))}{\sum_{j=1}^{n} \exp(\text{score}(h_{query}, h_j))}$$

3. **Vetor de Contexto ($c$):** 
   Calculamos a média ponderada de todos os estados ocultos originais usando os pesos obtidos:
   $$c = \sum_{i=1}^{n} \alpha_{i} h_i$$

Este vetor de contexto $c$ agora contém a informação focada da frase e será enviado para a camada linear final de classificação.

### Simulador Interativo do Mecanismo de Atenção
Execute a célula abaixo para abrir o simulador interativo direto no seu notebook. Escolha diferentes perfis de foco do modelo (Query) para observar como os scores, os pesos Softmax e a representação visual mudam dinamicamente para o título: **"Fed eleva taxa juros"**.

In [ ]:
from IPython.display import HTML

html_simulator = """
<div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f7f9fa; border: 1px solid #e1e8ed; border-radius: 8px; padding: 20px; max-width: 650px; margin: 10px auto; color: #2c3e50;">
    <h3 style="margin-top: 0; color: #1a5276; text-align: center;">⚙️ Simulador Interativo: Mecanismo de Atenção</h3>
    <p style="font-size: 13px; color: #566573; text-align: center;">Visualização em tempo real dos cálculos matemáticos e distribuição de pesos para o título: <b>"Fed eleva taxa juros"</b></p>
    
    <div style="margin: 15px 0; text-align: center;">
        <label style="font-weight: bold; font-size: 14px; margin-right: 10px;">Perfil de Intenção do Modelo (Query):</label>
        <select id="queryProfile" onchange="updateAttention()" style="padding: 6px 12px; border-radius: 4px; border: 1px solid #bdc3c7; background-color: white; font-size: 13px; cursor: pointer;">
            <option value="economia">Foco em Economia / Negócios (Alta afinidade com termos macroeconômicos)</option>
            <option value="acao">Foco em Ação / Dinâmica (Alta afinidade com o verbo principal)</option>
            <option value="neutro">Distribuição Neutra / Uniforme</option>
        </select>
    </div>

    <div style="margin: 25px 0;">
        <div style="display: flex; justify-content: space-between; text-align: center; font-weight: bold; font-size: 14px;">
            <div id="w0" style="flex: 1; padding: 15px 5px; margin: 3px; border-radius: 4px; transition: all 0.3s;">Fed</div>
            <div id="w1" style="flex: 1; padding: 15px 5px; margin: 3px; border-radius: 4px; transition: all 0.3s;">eleva</div>
            <div id="w2" style="flex: 1; padding: 15px 5px; margin: 3px; border-radius: 4px; transition: all 0.3s;">taxa</div>
            <div id="w3" style="flex: 1; padding: 15px 5px; margin: 3px; border-radius: 4px; transition: all 0.3s;">juros</div>
        </div>
    </div>

    <div style="background-color: white; border-radius: 6px; padding: 15px; font-size: 13px; border-left: 4px solid #3498db;">
        <b style="color: #2980b9; display: block; margin-bottom: 8px;">📊 Métricas Calculadas:</b>
        <table style="width: 100%; border-collapse: collapse; text-align: left;">
            <thead>
                <tr style="border-bottom: 1px solid #f2f2f2; color: #7f8c8d;">
                    <th style="padding: 5px 0;">Palavra</th>
                    <th>Score Bruto</th>
                    <th>Peso Softmax (α)</th>
                </tr>
            </thead>
            <tbody>
                <tr><td style="padding: 6px 0; font-weight: 600;">Fed</td><td id="s0">-</td><td id="p0" style="font-family: monospace; font-weight: bold;">-</td></tr>
                <tr><td style="padding: 6px 0; font-weight: 600;">eleva</td><td id="s1">-</td><td id="p1" style="font-family: monospace; font-weight: bold;">-</td></tr>
                <tr><td style="padding: 6px 0; font-weight: 600;">taxa</td><td id="s2">-</td><td id="p2" style="font-family: monospace; font-weight: bold;">-</td></tr>
                <tr><td style="padding: 6px 0; font-weight: 600;">juros</td><td id="s3">-</td><td id="p3" style="font-family: monospace; font-weight: bold;">-</td></tr>
            </tbody>
        </table>
        <div style="margin-top: 12px; padding-top: 8px; border-top: 1px dashed #e1e8ed; font-size: 12px; color: #7f8c8d; display: flex; justify-content: space-between;">
            <span>Σ Alphas = <span id="sumAlphas" style="font-weight: bold; color: #27ae60;">1.00</span></span>
            <span style="font-style: italic;">Vetor de contexto gerado com sucesso!</span>
        </div>
    </div>
</div>

<script>
function updateAttention() {
    var profile = document.getElementById("queryProfile").value;
    var data = {
        economia: { scores: [2.5, 0.4, 1.2, 2.9], weights: [0.38, 0.05, 0.10, 0.47] },
        acao: { scores: [0.2, 3.1, 0.5, 0.1], weights: [0.05, 0.81, 0.09, 0.05] },
        neutro: { scores: [1.0, 1.0, 1.0, 1.0], weights: [0.25, 0.25, 0.25, 0.25] }
    };

    var selected = data[profile];
    var colors = {
        economia: "rgba(26, 82, 118, ",
        acao: "rgba(211, 84, 0, ",
        neutro: "rgba(127, 140, 141, "
    };
    var textColors = {
        economia: "#1a5276", acao: "#d35400", neutro: "#7f8c8d"
    };

    for(var i=0; i<4; i++) {
        var scoreEl = document.getElementById("s" + i);
        var weightEl = document.getElementById("p" + i);
        var blockEl = document.getElementById("w" + i);
        
        scoreEl.innerText = selected.scores[i].toFixed(1);
        weightEl.innerText = (selected.weights[i] * 100).toFixed(0) + "%";
        
        var alpha = selected.weights[i];
        blockEl.style.backgroundColor = colors[profile] + (alpha * 0.9 + 0.1) + ")";
        blockEl.style.color = alpha > 0.35 ? "white" : textColors[profile];
        blockEl.style.border = "1px solid " + textColors[profile];
    }
}
// Inicializar
setTimeout(updateAttention, 200);
</script>
"""
HTML(html_simulator)

---
## Exercício 1: Calculando Pesos de Atenção Manualmente (20 min)

**Objetivo:** Compreender de forma prática e concreta os fundamentos algébricos do mecanismo de atenção sem abstrações de camadas prontas.

**Instruções para o Aluno:**
Você recebeu um tensor fictício de estados ocultos (`hidden_states`) correspondente às palavras de um título de notícia. Você também recebeu um vetor contendo a intenção de busca (`query`). Complete o código abaixo utilizando **PyTorch** para implementar as operações matemáticas que geram o vetor de contexto final.

In [ ]:
import torch
import torch.nn.functional as F

# Frase correspondente: ["Google", "anuncia", "nova", "IA"]
# Formato: [seq_len = 4, hidden_dim = 3]
hidden_states = torch.tensor([
    [0.8, 0.1, 0.2],  # Estado oculto extraído para 'Google'
    [0.1, 0.9, 0.1],  # Estado oculto extraído para 'anuncia'
    [0.2, 0.2, 0.1],  # Estado oculto extraído para 'nova'
    [0.9, 0.2, 0.9]   # Estado oculto extraído para 'IA'
], dtype=torch.float32)

# Vetor de consulta do modelo buscando focar em inovação tecnológica
query = torch.tensor([0.7, 0.1, 0.8], dtype=torch.float32)

# ======================================================================
# 1. TODO: Calcule os scores usando produto escalar (dot product)
# Dica: Multiplique a matriz de hidden_states pelo vetor query
scores = None

# 2. TODO: Aplique a função Softmax sobre os scores obtidos para extrair os alphas
attention_weights = None

# 3. TODO: Calcule o Vetor de Contexto consolidando a soma ponderada
context_vector = None
# ======================================================================

print("--- SEU RESULTADO ---")
print("Scores Brutos:", scores)
print("Pesos de Atenção (Alphas):", attention_weights)
if attention_weights is not None:
    print("Soma dos Alphas (Deve ser 1.0):", attention_weights.sum().item())
print("Vetor de Contexto Resultante:", context_vector)

---
## Exercício 2: Criando e Integrando a Camada de Atenção no PyTorch (20 min)

**Objetivo:** Construir uma arquitetura de rede neural robusta que processa batches de dados em paralelo aplicando o mecanismo de atenção.

**Instruções para o Aluno:**
Complete o esqueleto da classe `AttentionNewsClassifier`. Sua principal missão é extrair os estados da sequência fornecidos pela LSTM bidirecional, calcular os scores, aplicar o Softmax respeitando as dimensões tridimensionais do batch, e gerar o vetor de contexto usando multiplicação de matrizes em lote (`torch.bmm`).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionNewsClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # LSTM Bidirecional (Saída de cada passo terá dimensionalidade: hidden_dim * 2)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        
        # Camada Linear responsável pelo cálculo dinâmico do score
        self.attention_layer = nn.Linear(hidden_dim * 2, 1, bias=False)
        
        # Camada de saída totalmente conectada
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, text_tokens):
        # Entrada text_tokens: [batch_size, seq_len]
        embedded = self.embedding(text_tokens) # [batch_size, seq_len, embedding_dim]
        
        # lstm_out armazena os estados ocultos de todos os passos temporais
        # Formato de lstm_out: [batch_size, seq_len, hidden_dim * 2]
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # ======================================================================
        # 1. TODO: Passe o tensor lstm_out pela camada attention_layer e remova
        # a última dimensão utilizando .squeeze(-1). 
        # Saída esperada: [batch_size, seq_len]
        scores = None 
        
        # 2. TODO: Aplique a função Softmax na dimensão correta da sequência (dim=-1)
        # Saída esperada: [batch_size, seq_len]
        attn_weights = None 
        
        # 3. TODO: Gere a soma ponderada criando o vetor de contexto final.
        # Dica: Use o bloco abaixo para expandir dimensões e realizar o cálculo em lote (torch.bmm)
        if attn_weights is not None:
            attn_weights_unsqueezed = attn_weights.unsqueeze(1) # [batch_size, 1, seq_len]
            context = torch.bmm(attn_weights_unsqueezed, lstm_out).squeeze(1) # [batch_size, hidden_dim * 2]
        else:
            context = torch.zeros(text_tokens.size(0), lstm_out.size(2))
        # ======================================================================
        
        logits = self.fc(context)
        return logits, attn_weights

# Teste estrutural para verificação de formatos e dimensões
model = AttentionNewsClassifier(vocab_size=1000, embedding_dim=64, hidden_dim=128, num_classes=4)
mock_input = torch.randint(0, 1000, (2, 10)) # Batch de 2 títulos, contendo 10 palavras cada
logits, weights = model(mock_input)

print("--- ANÁLISE DE SHAPES ---")
print("Logits shape (Esperado [2, 4]):", logits.shape)
print("Pesos de Atenção shape (Esperado [2, 10]):", weights.shape if weights is not None else "Não implementado")

---
## Exercício 3: Interpretando e Visualizando Pesos com Heatmaps (20 min)

**Objetivo:** Compreender a maior vantagem prática do mecanismo de atenção: a capacidade de auditar e interpretar as decisões internas do modelo.

**Instruções para o Aluno:**
Complete a função de plotagem utilizando a biblioteca `seaborn` (`sns.heatmap`). O objetivo é plotar uma linha de mapa de calor onde cada coluna corresponda a uma palavra do título avaliado e o nível de intensidade de cor seja guiado pelo peso de atenção obtido do modelo.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def visualizar_atencao(titulo, pesos):
    """
    Parâmetros:
    titulo: Lista de strings contendo as palavras (Ex: ['ebola', 'outbreak', 'in', 'africa'])
    pesos: Tensor PyTorch ou Array Numpy com os pesos normalizados obtidos
    """
    if isinstance(pesos, torch.Tensor):
        pesos = pesos.detach().cpu().numpy()
        
    # Transforma o vetor em formato de matriz linha (1, seq_len) para o heatmap
    pesos_matriz = pesos.reshape(1, -1)
    
    plt.figure(figsize=(10, 2.5))
    
    # ======================================================================
    # TODO: Complete a chamada do heatmap do seaborn (sns.heatmap)
    # Parâmetros recomendados:
    # - data=pesos_matriz
    # - xticklabels=titulo (para nomear o eixo X com as palavras)
    # - yticklabels=[] (remover marcadores do eixo Y)
    # - annot=True (exibir os valores numéricos salvos dentro dos blocos)
    # - cmap='YlGnBu' (gradiente visual estético)
    # - cbar=False (ocultar barra lateral de legenda)
    # Seu código aqui
    
    # ======================================================================
    
    plt.title("Mapa de Calor dos Pesos de Atenção do Modelo")
    plt.tight_layout()
    plt.show()

# Exemplo de simulação de execução de teste
titulo_teste = ["fda", "approves", "new", "cancer", "drug"]
pesos_teste = torch.tensor([0.35, 0.05, 0.05, 0.45, 0.10])

visualizar_atencao(titulo_teste, pesos_teste)

---
## 💬 Pergunta para Discussão em Sala (Finalização - 10 min)

Ao plotar o mapa de calor acima com o título focado em saúde (*Health*): **"fda approves new cancer drug"**, o modelo atribuiu pesos altos ($0.35$ e $0.45$) para as palavras **"fda"** e **"cancer"**. 

1. Por que as palavras **"approves"** e **"new"** receberam pesos menores, mesmo sendo verbos e adjetivos importantes para a estrutura frasal?
2. Como esse comportamento comprova que a camada de atenção consegue mitigar ruídos e atuar como um filtro semântico eficiente para tarefas de classificação de texto?

In [ ]:
titulo_ambiguo = ["apple", "releases", "new", "trailer", "for", "sci-fi", "movie"]

# Simulando os pesos que um modelo bem treinado geraria:
# Note como "apple" tem um peso médio, mas "trailer" e "movie" roubam a cena
pesos_ambiguos = torch.tensor([0.15, 0.05, 0.02, 0.35, 0.01, 0.07, 0.35])

print("Categoria Real: Entretenimento (e)")
visualizar_atencao_gabarito(titulo_ambiguo, pesos_ambiguos)

## Criando embeddings usando Attention

Vamos fazer um estudo de caso comparando as soluções da aula do Word2Vec. O Objetivo é usar o mesmo dataset (UCI-News-Aggregator) e gerar embebdings, mas agora usando o mecanismo de Atenção. Para isso, precisamos de 4 etapas:

1. Pré-processamento: Carregar o CSV, limpar os títulos, construir um vocabulário e converter as palavras em números (índices).
2. DataLoaders: Criar lotes (batches) e preencher as frases com zeros (padding) para que todas tenham o mesmo tamanho.
3. O Modelo e Otimizador: Instanciar o AttentionNewsClassifier e configurar a função de perda (CrossEntropy) e o otimizador (Adam).
4. O Loop de Treino: Passar os dados pelo modelo, calcular o erro, ajustar os pesos (backpropagation) e medir a acurácia.

O código abaixo apresenta essa sequência:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from collections import Counter
import re
from sklearn.model_selection import train_test_split

# =====================================================================
# 1. PREPARAÇÃO DOS DADOS E VOCABULÁRIO
# =====================================================================

def clean_text(text):
    """Limpeza básica do texto: minúsculas e remoção de caracteres especiais."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

def build_vocab(text_series, max_vocab_size=10000):
    """Constrói o vocabulário mapeando palavras frequentes para índices."""
    all_words = []
    for tokens in text_series:
        all_words.extend(tokens)
        
    counter = Counter(all_words)
    # Deixa os índices 0 e 1 para PAD (Preenchimento) e UNK (Desconhecido)
    vocab = {'<PAD>': 0, '<UNK>': 1}
    
    for idx, (word, _) in enumerate(counter.most_common(max_vocab_size)):
        vocab[word] = idx + 2
        
    return vocab

# =====================================================================
# 2. DEFINIÇÃO DO DATASET (PyTorch)
# =====================================================================

class UCINewsDataset(Dataset):
    def __init__(self, titles, labels, vocab, max_len=25):
        self.titles = titles
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len
        
    def __len__(self):
        return len(self.titles)
        
    def __getitem__(self, idx):
        # Converte as palavras para índices numéricos
        tokens = self.titles.iloc[idx]
        indices = [self.vocab.get(w, self.vocab['<UNK>']) for w in tokens]
        
        # Padding: preenche com 0s se for menor, trunca se for maior
        if len(indices) < self.max_len:
            indices = indices + [self.vocab['<PAD>']] * (self.max_len - len(indices))
        else:
            indices = indices[:self.max_len]
            
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels.iloc[idx], dtype=torch.long)

# =====================================================================
# 3. O MODELO DE ATENÇÃO (Do Exercício 2)
# =====================================================================

import torch.nn.functional as F

class AttentionNewsClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attention_layer = nn.Linear(hidden_dim * 2, 1, bias=False)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, text_tokens):
        embedded = self.embedding(text_tokens) 
        lstm_out, _ = self.lstm(embedded)
        
        # Calcula a atenção
        scores = self.attention_layer(lstm_out).squeeze(-1)
        
        # MÁSCARA: Garante que os <PAD>s (zeros) não recebam atenção
        mask = (text_tokens == 0)
        scores = scores.masked_fill(mask, -1e9) # Joga o score para -infinito antes do Softmax
        
        attn_weights = F.softmax(scores, dim=-1)
        
        # Vetor de contexto
        attn_weights_unsqueezed = attn_weights.unsqueeze(1) 
        context = torch.bmm(attn_weights_unsqueezed, lstm_out).squeeze(1)
        
        logits = self.fc(context)
        return logits, attn_weights

# =====================================================================
# 4. FUNÇÃO PRINCIPAL DE TREINAMENTO
# =====================================================================

def train_model():
    print("Iniciando pipeline de treinamento...")
    
    
    df = pd.read_csv("uci-news-aggregator.csv")
    
    # Mapeamento de categorias (b: Business, t: Tech, e: Entertainment, m: Health)
    cat_map = {'b': 0, 't': 1, 'e': 2, 'm': 3}
    df['LABEL'] = df['CATEGORY'].map(cat_map)
    
    # Processa os textos
    df['TOKENS'] = df['TITLE'].apply(clean_text)
    
    # Cria o vocabulário
    vocab = build_vocab(df['TOKENS'])
    vocab_size = len(vocab)
    print(f"Tamanho do vocabulário: {vocab_size} palavras")
    
    # Configura Dataset e DataLoader
    dataset = UCINewsDataset(df['TOKENS'], df['LABEL'], vocab)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    # Instancia o Modelo
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = AttentionNewsClassifier(vocab_size=vocab_size, embedding_dim=64, hidden_dim=64, num_classes=4).to(device)
    
    # Função de perda e Otimizador
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Loop de Treinamento
    epochs = 3
    print("\n--- Iniciando o Treinamento ---")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct_predictions = 0
        
        for batch_texts, batch_labels in dataloader:
            batch_texts, batch_labels = batch_texts.to(device), batch_labels.to(device)
            
            # Zera os gradientes
            optimizer.zero_grad()
            
            # Forward pass
            logits, _ = model(batch_texts)
            
            # Calcula a perda
            loss = criterion(logits, batch_labels)
            
            # Backward pass e Otimização
            loss.backward()
            optimizer.step()
            
            # Acumula métricas
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct_predictions += (preds == batch_labels).sum().item()
            
        epoch_loss = total_loss / len(dataloader)
        epoch_acc = correct_predictions / len(dataset)
        
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f}")
        
    print("\nTreinamento Finalizado!")
    return model, vocab

In [ ]:
model, vocab = train_model()

### Sugestão de desafio: 

Treine um modelo SVM usando o embedding do Attention como modelo e compare com o resultado Word2Vec. Voce pode seguir esse pipeline: 

1. criar batches para receber os textos que precisam ser transformados em números usando o embedding do Attention
2. passar os batches pela camada de Attention e pela camada LSTM
3. Calcular atenção
4. utilize unsqueeze na saída da aplicação da softmax para obter as features da amostra
5. depois que fizer para todas as amostras concatene tudo em uma matriz (features) e um vetor (labels)
6. com essa saída (features, labels) treine um modelo SVM
7. Use 
```python 
X_train, X_test, y_train, y_test = train_test_split(X[0:100000], y[0:100000], test_size=0.3,random_state=42)
```


In [ ]:
df = pd.read_csv("uci-news-aggregator.csv")
    
# Mapeamento de categorias (b: Business, t: Tech, e: Entertainment, m: Health)
cat_map = {'b': 0, 't': 1, 'e': 2, 'm': 3}
df['LABEL'] = df['CATEGORY'].map(cat_map)

# Processa os textos
df['TOKENS'] = df['TITLE'].apply(clean_text)

# Cria o vocabulário
vocab = build_vocab(df['TOKENS'])
vocab_size = len(vocab)
print(f"Tamanho do vocabulário: {vocab_size} palavras")

# Configura Dataset e DataLoader
dataset = UCINewsDataset(df['TOKENS'], df['LABEL'], vocab)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)